<div style="padding: 20px 24px; border-radius: 12px; background: linear-gradient(110deg, #12304a, #187b8f); color: white; margin-bottom: 16px;">
  <div style="font-size: 0.78em; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.85;">HydroMT-SFINCS | Jakarta 03</div>
  <h1 style="color: white; margin: 8px 0 4px;">Run the SFINCS model: Jakarta Bay 2026</h1>
  <div style="opacity: 0.95;">Launch the compiled SFINCS model and follow its run log.</div>
</div>


SFINCS is a compiled Fortran binary — Python cannot execute it directly.  
This notebook shows two ways to launch a simulation:

| Method | Best for | Output log |
|--------|----------|------------|
| **Batch file** (`run.bat`) | Double-click from Explorer; share with colleagues who don't use Python | `sfincs.log` |
| **Python** (`run_sfincs`) | Running from within this notebook; streams live output into the cell | `sfincs_log.txt` |

Both methods call the same executable and produce identical results.

<div style="padding: 14px 18px; border-left: 4px solid #d99000; background: #fff7e6; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong>Prerequisites:</strong> Download the SFINCS executable from https://download.deltares.nl/en/download/sfincs/</div>

---
## Settings

In [1]:
from pathlib import Path

# ── Adjust these two paths ────────────────────────────────────────────────────
model_root = Path("../model/jakarta_2026_tanpa_struktur")  # [JAKARTA] folder containing sfincs.inp
sfincs_exe_dir = Path(r"F:\BTP\SFINCS_2026_01_release\SFINCS_v2.4.0_Galibier_release_exe")  # [JAKARTA] folder containing sfincs.exe
# ─────────────────────────────────────────────────────────────────────────────

sfincs_exe = sfincs_exe_dir / "sfincs.exe"  # full path — used by run_sfincs

print(f"Model folder  : {model_root.resolve()}")
print(f"Executable    : {sfincs_exe.resolve()}")
print(f"Exe found     : {sfincs_exe.exists()}")

Model folder  : F:\asemelehoy\skema_terbaik\model\jakarta_2026_tanpa_struktur
Executable    : F:\BTP\SFINCS_2026_01_release\SFINCS_v2.4.0_Galibier_release_exe\sfincs.exe
Exe found     : True


<div style="padding: 14px 18px; border-left: 4px solid #187b8f; background: #eaf6f8; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;">
  <strong><span style="font-size: 1.15em; margin-right: 4px;">&#128161;</span> [JAKARTA] Note</strong>
  <p style="margin: 8px 0 0;">One run (24 Oct 00:00 to 30 Oct 23:00, 120 426 cells) took 67 minutes in the cloud with 2 CPU threads (two runs side by side on 4 cores). Without Windows, the same model runs with the Docker image of Deltares: <code>docker run --rm -v &lt;full path to model/jakarta_2026&gt;:/data deltares/sfincs-cpu:latest</code>.</p>
</div>

---
## Method 1 — Batch file

A batch file (`run.bat`) is a two-line Windows script: it sets one environment variable
and calls the SFINCS binary. No Python is needed at runtime — useful for sharing a simulation
with colleagues or scheduling automated runs.

The recommended way to write it is via `SfincsModel.write_batch_file()`, which handles both
Windows (`run.bat`) and Linux/macOS (`run.sh`, with execute permissions set automatically).

> **Path convention:** `SfincsModel` expects the **folder** containing the binary (`exe_path`),
> not the full path to the executable itself.

In [2]:
from hydromt_sfincs import SfincsModel

# Open the model read-only and pass the executable folder
sf = SfincsModel(
    root=model_root,
    mode="r",
    exe_path=str(sfincs_exe_dir),  # folder — SfincsModel appends sfincs.exe internally
)

bat_file = sf.write_batch_file()
print(f"Launcher written : {bat_file}")
print(f"\nContents:\n{bat_file.read_text()}")

C:\Users\ASUS\miniconda3\envs\asemelehoy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Launcher written : F:\asemelehoy\skema_terbaik\model\jakarta_2026_tanpa_struktur\run.bat

Contents:
set HDF5_USE_FILE_LOCKING=FALSE
F:\BTP\SFINCS_2026_01_release\SFINCS_v2.4.0_Galibier_release_exe\sfincs.exe



> **Alternative — write the batch file manually** (no model object needed):
> ```python
> bat_file = model_root / 'run.bat'
> bat_file.write_text(
>     f'set HDF5_USE_FILE_LOCKING=FALSE\n"{sfincs_exe}"\n',
>     encoding='ascii',
> )
> ```

### Launching the batch file

| Interface | Command |
|-----------|---------|
| **Windows Explorer** | Navigate to the model folder → double-click `run.bat` |
| **Command Prompt** | `cd <model_folder>` then `run.bat` |
| **PowerShell / VS Code terminal** | `cd <model_folder>` then `.\run.bat` |

A Command Prompt window opens and streams SFINCS output while it runs.
`sfincs.log` is written to the model folder on completion.

> **HPC / network drive:** `HDF5_USE_FILE_LOCKING=FALSE` prevents NetCDF write errors
> caused by file-locking restrictions on some shared file systems.

---
## Method 2 — Python (`run_sfincs`)

`run_sfincs` from `hydromt_sfincs.run` launches the executable as a subprocess and:

- streams output **live** into the notebook cell
- raises a `RuntimeError` immediately if the run fails (non-zero exit code)
- writes all captured output to `sfincs_log.txt` in the model folder

> **Path convention:** `run_sfincs` expects the **full path** to the executable
> (`sfincs_exe_dir / 'sfincs.exe'`), unlike `SfincsModel` which takes the folder.

In [ ]:
from hydromt_sfincs.run import run_sfincs

if sfincs_exe.exists():
    run_sfincs(
        model_root=model_root,
        sfincs_exe=sfincs_exe,  # full path to sfincs.exe
    )
else:
    print(f"Skipping: executable not found at {sfincs_exe}")
    print("Download from: https://download.deltares.nl/en/download/sfincs/")

>> F:\BTP\SFINCS_2026_01_release\SFINCS_v2.4.0_Galibier_release_exe\sfincs.exe


------------ Welcome to SFINCS ------------

  @@@@@  @@@@@@@ @@ @@  @@   @@@@   @@@@@
 @@@ @@@ @@@@@@@ @@ @@@ @@ @@@@@@@ @@@ @@@
 @@@     @@      @@ @@@ @@ @@   @@ @@@
  @@@@@  @@@@@@  @@ @@@@@@ @@       @@@@@
     @@@ @@      @@ @@ @@@ @@   @@     @@@
 @@@ @@@ @@      @@ @@  @@  @@@@@@ @@@ @@@
  @@@@@  @@      @@ @@   @   @@@@   @@@@@

              ..............
          ......:@@@@@@@@:......
       ..::::..@@........@@.:::::..
     ..:::::..@@..::..::..@@.::::::..
    .::::::..@@............@@.:::::::.
   .::::::..@@..............@@.:::::::.
  .::::::::..@@............@@..::::::::.
 .:::::::::...@@.@..@@..@.@@..::::::::::.
 .:::::::::...:@@@..@@..@@@:..:::::::::..
 ............@@.@@..@@..@@.@@............
 ^^^~~^^~~^^@@..............@@^^^~^^^~~^^
 .::::::::::@@..............@@.:::::::::.
  .......:.@@.....@.....@....@@.:.......
   .::....@@......@.@@@.@....@@.....::.
    .:::~@@.:...:.@@...@@.:.:.@@

---
## Check the run log

The log location depends on how the model was launched:

| Method | Log file |
|--------|----------|
| Batch file | `sfincs.log` (written by the SFINCS binary) |
| `run_sfincs` | `sfincs_log.txt` (captured stdout/stderr) |

The last lines should contain `Simulation finished`.

In [ ]:
# run_sfincs writes sfincs_log.txt; the SFINCS binary itself writes sfincs.log
log_file = model_root / "sfincs_log.txt"
if not log_file.exists():
    log_file = model_root / "sfincs.log"

if log_file.exists():
    lines = log_file.read_text(errors="replace").splitlines()
    print(f"Log: {log_file.name}  ({len(lines)} lines)")
    print("\n--- Last 20 lines ---")
    print("\n".join(lines[-20:]))
else:
    print("No log file found — has the model run yet?")

---
## Output files

A successful SFINCS run produces two NetCDF files in the model folder:

| File | Contents |
|------|----------|
| `sfincs_map.nc` | Gridded results: water levels (`zs`), max water level (`zsmax`), wave height (`hm0`) |
| `sfincs_his.nc` | Time series at observation points: water level, wave height, wave direction |

In [ ]:
for fn in ["sfincs_map.nc", "sfincs_his.nc"]:
    fp = model_root / fn
    if fp.exists():
        print(f"  {fn}: {fp.stat().st_size / 1e6:.1f} MB")
    else:
        print(f"  {fn}: NOT FOUND")

---
## Next steps

Continue to the validation of the water levels against the SKLP tide gauge:

- [Validate the water levels](04_validasi_muka_air.ipynb)